In [10]:
from typing import Any, List, Callable, Union
import h5py
import pickle
from pathlib import Path
import numpy as np
from scipy.stats import pearsonr
import torch
from enformer_pytorch import Enformer
from enformer_pytorch import from_pretrained
from enformer_pytorch.finetune import HeadAdapterWrapper
from transformers import get_scheduler
from torch.utils.data import TensorDataset, DataLoader

from torch.optim import AdamW
from tqdm.auto import tqdm

import sys
sys.path.append('../src')
import enformer_dataloader_np
from enformer_dataloader_np import NumpyDataModule

# use GPU 1:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# blah

# Load data

In [11]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda')

In [20]:
import os
import shutil
import numpy as np
import numpy as np

def train_test_split(arr, test_size=0.25, random_state=None):
    """
    Split arrays or matrices into random train and test subsets.
    
    Args:
        arr: Array to be split
        test_size: Proportion of the dataset to include in the test split
        random_state: Seed used by the random number generator
        
    Returns:
        train: Training subset
        test: Testing subset
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    n = len(arr)
    indices = np.random.permutation(n)
    test_size = int(n * test_size)
    
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]
    
    return arr[train_indices], arr[test_indices]


def create_dataset_splits(data_dir, train_size=0.7, valid_size=0.15, test_size=0.15, seed=42):
    """
    Create train/valid/test splits from existing test files.
    
    Args:
        data_dir: Directory containing the test_*.npz files
        train_size: Proportion for training set
        valid_size: Proportion for validation set
        test_size: Proportion for test set
        seed: Random seed for reproducibility
        
    Returns:
        Dictionary with train, valid, test file lists
    """
    # Get all test files
    files = [f for f in os.listdir(data_dir) if f.startswith('test_') and f.endswith('.npz')]
    files = sorted(files)  # Sort for reproducibility
    
    # Create file indices
    indices = np.arange(len(files))
    
    # First split: train and temp (valid+test combined)
    train_indices, temp_indices = train_test_split(
        indices, test_size=valid_size+test_size, random_state=seed
    )
    
    # Second split: valid and test from temp
    valid_size_adjusted = valid_size / (valid_size + test_size)
    valid_indices, test_indices = train_test_split(
        temp_indices, test_size=(1-valid_size_adjusted), random_state=seed
    )
    
    # Create lists of files for each split
    train_files = [os.path.join(data_dir, files[i]) for i in train_indices]
    valid_files = [os.path.join(data_dir, files[i]) for i in valid_indices]
    test_files = [os.path.join(data_dir, files[i]) for i in test_indices]
    
    print(f"Created splits from {len(files)} files:")
    print(f"  Train: {len(train_files)} files ({len(train_files)/len(files):.1%})")
    print(f"  Valid: {len(valid_files)} files ({len(valid_files)/len(files):.1%})")
    print(f"  Test: {len(test_files)} files ({len(test_files)/len(files):.1%})")
    
    return {
        'train': train_files,
        'valid': valid_files,
        'test': test_files
    }

# Usage example
human_splits = create_dataset_splits(
    '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/human'
)
mouse_splits = create_dataset_splits(
    '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/mouse'
)

def move_files_to_splits(data_dir, output_dir, train_size=0.7, valid_size=0.15, test_size=0.15, seed=42, copy=True):
    """
    Move or copy files into train/valid/test directories
    
    Args:
        data_dir: Directory containing the test_*.npz files
        output_dir: Base directory to create train/valid/test subdirectories
        train_size: Proportion for training set
        valid_size: Proportion for validation set
        test_size: Proportion for test set
        seed: Random seed for reproducibility
        copy: If True, copy files instead of moving them
    """
    # Get all test files
    files = [f for f in os.listdir(data_dir) if f.startswith('test_') and f.endswith('.npz')]
    files = sorted(files)  # Sort for reproducibility
    
    # Create output directories
    train_dir = os.path.join(output_dir, 'train')
    valid_dir = os.path.join(output_dir, 'valid')
    test_dir = os.path.join(output_dir, 'test')
    
    for d in [train_dir, valid_dir, test_dir]:
        os.makedirs(d, exist_ok=True)
    
    # Split the files
    splits = create_dataset_splits(data_dir, train_size, valid_size, test_size, seed)
    
    # Function to extract filename from path
    def get_filename(path):
        return os.path.basename(path)
    
    # Move or copy files
    operation = shutil.copy2 if copy else shutil.move
    
    # Process each split
    for train_file in splits['train']:
        dest = os.path.join(train_dir, get_filename(train_file))
        operation(train_file, dest)
        
    for valid_file in splits['valid']:
        dest = os.path.join(valid_dir, get_filename(valid_file))
        operation(valid_file, dest)
        
    for test_file in splits['test']:
        dest = os.path.join(test_dir, get_filename(test_file))
        operation(test_file, dest)
    
    print(f"Files {'copied' if copy else 'moved'} to:")
    print(f"  Train: {train_dir} ({len(splits['train'])} files)")
    print(f"  Valid: {valid_dir} ({len(splits['valid'])} files)")
    print(f"  Test: {test_dir} ({len(splits['test'])} files)")

# Usage
move_files_to_splits(
    '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/human',
    '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human',
    copy=True  # Set to False if you want to move instead of copy
)

move_files_to_splits(
    '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/mouse',
    '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse',
    copy=True
)

Created splits from 1937 files:
  Train: 1356 files (70.0%)
  Valid: 291 files (15.0%)
  Test: 290 files (15.0%)
Created splits from 2017 files:
  Train: 1412 files (70.0%)
  Valid: 303 files (15.0%)
  Test: 302 files (15.0%)
Created splits from 1937 files:
  Train: 1356 files (70.0%)
  Valid: 291 files (15.0%)
  Test: 290 files (15.0%)
Files copied to:
  Train: /home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/train (1356 files)
  Valid: /home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/valid (291 files)
  Test: /home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/test (290 files)
Created splits from 2017 files:
  Train: 1412 files (70.0%)
  Valid: 303 files (15.0%)
  Test: 302 files (15.0%)
Files copied to:
  Train: /home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/train (1412 files)
  Valid: /home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_

# Retrieve embeddings

In [12]:
import torch
import pytorch_lightning as pl
from enformer_pytorch import Enformer
from enformer_pytorch.finetune import HeadAdapterWrapper

class EnformerWithEmbeddings(pl.LightningModule):
    def __init__(self, num_tracks=25, target_layer='transformer.layers.5'):
        super().__init__()
        self.enformer = Enformer.from_pretrained('EleutherAI/enformer-official-rough')
        self.model = HeadAdapterWrapper(
            enformer=self.enformer,
            num_tracks=num_tracks,
            post_transformer_embed=False
        )
        
        # Load the pretrained weights
        # checkpoint = torch.load(model_path)
        # self.model.load_state_dict(checkpoint['model_state_dict'])
        
        self.target_layer = target_layer
        self.hook_store = {}
        self.set_hooks()
        
    def set_hooks(self):
        """Set up hooks to capture embeddings from target layer"""
        def get_activation(name):
            def hook(module, input, output):
                self.hook_store[name] = output.detach()
            return hook
        
        # Navigate to the target layer and register the hook
        layer_parts = self.target_layer.split('.')
        target = self.model.enformer
        for part in layer_parts:
            target = getattr(target, part)
        
        target.register_forward_hook(get_activation(self.target_layer))
        
    def get_embeddings(self, x):
        """Extract embeddings from the target layer and also return model predictions
        
        Returns:
            tuple: (embeddings, predictions) where embeddings are from the target layer
                  and predictions are the full model output
        """
        self.eval()
        with torch.no_grad():
            # Run a forward pass to trigger the hooks and get predictions
            predictions = self.model(x)
            # Return the captured embeddings and predictions
            return self.hook_store[self.target_layer], predictions
    
    def forward(self, x, target=None):
        preds = self.model(x)
        if target is None:
            return preds
        return self.model(seq=x, target=target)
    

# Example usage:
# model.enformer.conv_tower.5.2.to_attn_logits
# transformer.10.1.fn.4
model = EnformerWithEmbeddings(target_layer='conv_tower.5.2.to_attn_logits')
input_tensor = torch.randint(0, 5, (1, 196_608)) # for ACGTN, in that order (-1 for padding)
input_tensor.shape
outs = model.get_embeddings(input_tensor)

In [21]:
#!/usr/bin/env python
import os
import argparse
import torch
import numpy as np
from tqdm import tqdm
import pytorch_lightning as pl
from enformer_pytorch import Enformer
from enformer_pytorch.finetune import HeadAdapterWrapper

# Import the NumpyDataModule class - assuming it's in your src directory
import sys
sys.path.append('src')
import enformer_dataloader_np
from enformer_dataloader_np import NumpyDataModule  # If your class definition is in this file

class EnformerWithEmbeddings(pl.LightningModule):
    def __init__(self, num_tracks=5313, target_layer='transformer.layers.5'):
        super().__init__()
        self.enformer = Enformer.from_pretrained('EleutherAI/enformer-official-rough')
        self.model = HeadAdapterWrapper(
            enformer=self.enformer,
            num_tracks=num_tracks,
            post_transformer_embed=False
        )
        
        self.target_layer = target_layer
        self.hook_store = {}
        self.set_hooks()
        
    def set_hooks(self):
        """Set up hooks to capture embeddings from target layer"""
        def get_activation(name):
            def hook(module, input, output):
                self.hook_store[name] = output.detach()
            return hook
        
        # Navigate to the target layer and register the hook
        layer_parts = self.target_layer.split('.')
        target = self.model.enformer
        for part in layer_parts:
            target = getattr(target, part)
        
        target.register_forward_hook(get_activation(self.target_layer))
        
    def get_embeddings(self, x):
        """Extract embeddings from the target layer and also return model predictions
        
        Returns:
            tuple: (embeddings, predictions) where embeddings are from the target layer
                  and predictions are the full model output
        """
        self.eval()
        with torch.no_grad():
            # Run a forward pass to trigger the hooks and get predictions
            predictions = self.model(x)
            # Return the captured embeddings and predictions
            return self.hook_store[self.target_layer], predictions
    
    def forward(self, x, target=None):
        preds = self.model(x)
        if target is None:
            return preds
        return self.model(seq=x, target=target)


def process_dataset(data_loader, model, output_dir, device='cuda', max_batches=None):
    """Process all batches in a dataset and save embeddings to disk"""
    os.makedirs(output_dir, exist_ok=True)
    
    batch_count = 0
    with torch.no_grad():
        for batch_idx, (sequences, targets) in enumerate(tqdm(data_loader, desc="Processing Batches")):
            if max_batches is not None and batch_idx >= max_batches:
                break
                
            # Move data to device
            sequences = sequences.to(device)
            
            # Get embeddings and predictions
            embeddings, predictions = model.get_embeddings(sequences)
            
            # Save embeddings and predictions for each sample in the batch
            for i in range(sequences.shape[0]):
                # Create a unique identifier for this sample
                sample_id = batch_idx * data_loader.batch_size + i
            
            batch_count += 1
    
    print(f"Processed {batch_count} batches, saved embeddings to {output_dir}")


human_data = '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/human'
mouse_data = '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/mouse'
            
output_dir = '/home/rajesh/projects/hackathon/SAE_Hackathon/results/enformer_embeddings'
target_layer = 'conv_tower.5.2.to_attn_logits'
batch_size = 4
max_batches = None
gpu = True  

# Set device
device = 'cuda' if gpu and torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Initialize the model
target_layer = 'conv_tower.5.2.to_attn_logits'
model = EnformerWithEmbeddings(target_layer=target_layer)
model = model.to(device)
model.eval()

# Create output directories
human_output_dir = os.path.join(output_dir, 'human')
mouse_output_dir = os.path.join(output_dir, 'mouse')

# Initialize data modules
human_data_module = NumpyDataModule(
    train_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/train",
    val_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/valid",
    test_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/test",
    batch_size=batch_size
)

mouse_data_module = NumpyDataModule(
    train_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/train",
    val_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/valid",
    test_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/test",
    batch_size=batch_size
)

# Setup data modules
human_data_module.setup()
mouse_data_module.setup()

# Get data loaders
human_loader = human_data_module.test_dataloader()
mouse_loader = mouse_data_module.test_dataloader()

# Process human dataset
print("Processing human dataset...")
process_dataset(human_loader, model, human_output_dir, device, max_batches)

# Process mouse dataset
print("Processing mouse dataset...")
process_dataset(mouse_loader, model, mouse_output_dir, device, max_batches)

print("Embedding extraction complete!")


Using device: cuda
The batch limit is: None
Found 1356 NumPy files
Found 291 NumPy files
Found 290 NumPy files
The batch limit is: None
Found 1412 NumPy files
Found 303 NumPy files
Found 302 NumPy files
Processing human dataset...


Processing Batches: 100%|██████████| 73/73 [00:30<00:00,  2.36it/s]


Processed 73 batches, saved embeddings to /home/rajesh/projects/hackathon/SAE_Hackathon/results/enformer_embeddings/human
Processing mouse dataset...


Processing Batches: 100%|██████████| 76/76 [00:32<00:00,  2.35it/s]

Processed 76 batches, saved embeddings to /home/rajesh/projects/hackathon/SAE_Hackathon/results/enformer_embeddings/mouse
Embedding extraction complete!
